In [ ]:
import pandas as pd
import re
import numpy as np



df = pd.read_csv("PEN_Combined_21-25_Final.csv")


print("Shape:", df.shape)
display(df.head())
print("\nColumns:", list(df.columns))

In [ ]:
#extract each "type" of reason from the genre column
GENRE_COL = "Genre"
def split_genres(x):
    if pd.isna(x):
        return []
    return [g.strip().upper() for g in re.split(r"[;,/]", str(x)) if g.strip()]

df["genres_list"] = df[GENRE_COL].apply(split_genres)
df["genres_list"].head(10)


In [ ]:
#create keys
GENRE_TO_CHALLENGE_REASON = {
    # Identity & sexuality
    "LGBT": "LGBTQ+ identity & sexuality",
    "QUEER": "LGBTQ+ identity & sexuality",

    # Race & power
    "RACE": "Race, racism, or systemic inequality",

    # Gender & politics
    "FEMINISM": "Gender politics / feminism",

    # Sexual / romantic themes
    "ROMANCE": "Romantic or sexual themes",

    # Psychological content
    "MENTAL HEALTH": "Mental health, trauma, or self-harm",

    # Tone / darkness (often linked to maturity concerns)
    "DARK": "Mature or disturbing themes",

    # Age-related framing
    "YOUNG ADULT": "Age-appropriateness concerns",
    "COMING OF AGE": "Age-appropriateness concerns",
}


In [ ]:
def infer_challenge_reasons(genres):
    reasons = set()
    for g in genres:
        if g in GENRE_TO_CHALLENGE_REASON:
            reasons.add(GENRE_TO_CHALLENGE_REASON[g])
    return list(reasons)

df["challenge_reasons"] = df["genres_list"].apply(infer_challenge_reasons)
df[["genres_list", "challenge_reasons"]].head(10)


In [ ]:
#What challenge reasons?
sorted({r for lst in df["challenge_reasons"] for r in lst})


In [ ]:
#How many books per challenge reason?
reason_counts = (
    df.explode("challenge_reasons")
      .dropna(subset=["challenge_reasons"])
      .groupby("challenge_reasons")
      .size()
      .sort_values(ascending=False)
)

reason_counts


In [ ]:
#Are there books with multiple challenge reasons?
df["num_reasons"] = df["challenge_reasons"].apply(len)

multi_reason_books = df[df["num_reasons"] >= 2]
multi_reason_books.shape[0]


In [ ]:
#Examples of such books
multi_reason_books[
    ["genres_list", "challenge_reasons", "num_reasons"]
].head(10)



In [ ]:
#What was the most common challenge reason?
reason_counts.idxmax(), reason_counts.max()


In [ ]:
#Unclear categories:

(df["num_reasons"] >= 2).mean() * 100




In [ ]:
#Overlapping categories
(df["num_reasons"] == 0).mean() * 100


In [ ]:
#Because the dataset does not include an explicit “challenge reason” variable, we inferred potential reasons from the genre labels provided. 
#Genres such as LGBT, Queer, Race, Mental Health, Feminism, and Romance correspond to themes that are frequently contested in school book challenges.
#This approach reveals that challenges are often driven by overlapping concerns—particularly around identity, sexuality, race, and mental health—rather than a single isolated issue.




